# Applicazione di ELIta al corpus r/Italia — keyword *notizie*, *film*, *sport*

Questo notebook applica il lessico ELIta (originale e versioni ricalcolate) ai commenti raccolti da r/**Italia** con keyword: **notizie**, **film** e **sport**.

## Import e configurazione

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

from Fase3.support import (
    BASIC_EMOTIONS, SEVEN_EMOTIONS, POSITIVE, NEGATIVE,
    load_corpus, load_all_matrices,
    detect_emotions, apply_corpus_mean_norm,
    plot_emotion_bars, plot_emotion_grid,
)

CORPUS_CSV   = Path('corpus_Italia_multi.csv')
TOKENS_CSV   = Path('tokens_Italia_multi.csv')
ELITA_CSV    = Path('../Fase1/ELIta_INTENSITY_Matrix.csv')
ALPHA_02_CSV = Path('../Fase2/output_csv/elita_recalculated_0_2.csv')
ALPHA_05_CSV = Path('../Fase2/output_csv/elita_recalculated_0_5.csv')
ALPHA_08_CSV = Path('../Fase2/output_csv/elita_recalculated_0_8.csv')
OUTPUT_DIR   = Path('output_confronto')
OUTPUT_DIR.mkdir(exist_ok=True)

print('Configurazione caricata.')

Configurazione caricata.


## Caricamento corpus, token e matrici ELIta

In [2]:
df_corpus, df_tokens = load_corpus(CORPUS_CSV, TOKENS_CSV)
print('Corpus:', len(df_corpus))
print('Token:', len(df_tokens))
print('Post:', (df_corpus['type'] == 'post').sum())
print('Commenti:', (df_corpus['type'] == 'comment').sum())

Corpus: 4311
Token: 146186
Post: 284
Commenti: 4027


In [3]:
MATRICES = load_all_matrices(ELITA_CSV, ALPHA_02_CSV, ALPHA_05_CSV, ALPHA_08_CSV)
df_elita_orig = MATRICES['Originale (α=0)']
print('Matrici:', list(MATRICES.keys()))

Matrici: ['Originale (α=0)', 'Ibrido (α=0.2)', 'Ibrido (α=0.5)', 'Ibrido (α=0.8)']


## Funzione base e prima analisi (raw)

Per ogni commento/post si sommano i vettori emotivi di tutti i lemmi ADJ+NOUN+VERB+EMOJI trovati in ELIta.

Nessun filtro, nessuna normalizzazione. Viene usato elita_orig (α=0) per questa prima analisi.

In [4]:
df_raw     = detect_emotions(df_corpus, df_tokens, df_elita_orig)
counts_raw = df_raw['dominant_emotion'].value_counts()
total      = len(df_raw)

df_raw.to_csv(OUTPUT_DIR / 'notizie_emotion_results_raw.csv', index=False)
print(f'Salvato: {OUTPUT_DIR / "notizie_emotion_results_raw.csv"}')

print('Distribuzione emozione dominante — raw:')
for e in BASIC_EMOTIONS + ['neutrale']:
    n = counts_raw.get(e, 0)
    print('{:<15s} {:>4d} ({:>4.1f}%) {}'.format(e, n, n/total*100, '█'*int(n/total*40)))

Salvato: output_confronto/notizie_emotion_results_raw.csv
Distribuzione emozione dominante — raw:
gioia            666 (15.4%) ██████
aspettativa     2569 (59.6%) ███████████████████████
rabbia           163 ( 3.8%) █
disgusto          17 ( 0.4%) 
tristezza        175 ( 4.1%) █
sorpresa          68 ( 1.6%) 
paura            214 ( 5.0%) █
fiducia          154 ( 3.6%) █
neutrale         285 ( 6.6%) ██


In [5]:
plot_emotion_bars(counts_raw, total, 'Distribuzione emozione dominante — analisi raw').show()

Aspettativa domina massicciamente (~60%).

Si deve capire se è un bias legato alle keyword di ricerca e al dominio, o se come accennato in ItEm è dovuto al fatto che aspettativa sia un'emozione più "generica" e che abbia creato confusione durante la costruzione del lessico.

## 1° passo: rimozione di aspettativa (corpus_7emo)

Come `corpus_sei_emo` in ItEm (che escludeva fiducia e attese), escludiamo aspettativa per vedere la struttura emotiva sottostante.

In [6]:
df_7emo  = detect_emotions(df_corpus, df_tokens, df_elita_orig, emotions=SEVEN_EMOTIONS)
counts_7 = df_7emo['dominant_emotion'].value_counts()

plot_emotion_grid(
    panels=[
        ('Raw (8 emozioni)',df_raw),
        ('Senza aspettativa (7 emozioni)',df_7emo),
    ],
    total=total,
    title='Raw vs corpus_7emo (senza aspettativa)',
    emotions=BASIC_EMOTIONS,
    height=400,
).show()

Senza aspettativa emerge massivamente gioia da 19% passa al 50%. Non risolve il problema, tolta un'emozione dominate ne esce un'altra, ma è utile per esplorare la struttura emotiva del dominio.

L'approccio da applicare è però quello di **corpus_mean** di ItEm (Formula 3.5), che normalizza i punteggi dopo averli accumulati, ridimensionando le emozioni sistematicamente alte.

## 2° passo - Corpus_mean di ItEm (Formula 3.5)

In ItEm il **corpus_mean** normalizza **dopo** aver accumulato i punteggi grezzi di ogni documento. La logica è che se un'emozione ha un valore medio alto su tutto il corpus (bias sistematico), dividerla per la sua media la riporta su una scala comparabile con le altre emozioni.

In [7]:
df_corpus_mean, mu_raw = apply_corpus_mean_norm(df_raw)
counts_cm = df_corpus_mean['dominant_emotion'].value_counts()

display(pd.Series(mu_raw, name='μ_e').round(4).to_frame())

,μ_e
gioia,4.3504
aspettativa,5.2944
rabbia,2.7043
disgusto,1.4988
tristezza,2.8587
sorpresa,3.2170
paura,3.1181
fiducia,3.9189


In [8]:
print('Distribuzione — corpus_mean ItEm (Formula 3.5):')
df_dist = pd.DataFrame([{
    'emozione': e,
    'n (corpus_mean)': counts_cm.get(e, 0),
    '%': round(counts_cm.get(e, 0) / total * 100, 1),
    'Δ vs raw': counts_cm.get(e, 0) - counts_raw.get(e, 0),
} for e in BASIC_EMOTIONS + ['neutrale']]).set_index('emozione')

display(df_dist)

Distribuzione — corpus_mean ItEm (Formula 3.5):


,n (corpus_mean),%,Δ vs raw
emozione,,,
gioia,641,14.9,-25
aspettativa,310,7.2,-2259
rabbia,340,7.9,177
disgusto,858,19.9,841
tristezza,454,10.5,279
sorpresa,519,12.0,451
paura,367,8.5,153
fiducia,537,12.5,383
neutrale,285,6.6,0


In [9]:
results_raw_versions = {}
results_cm_ibridi    = {}
for vname, df_e in MATRICES.items():
    df_raw_v = detect_emotions(df_corpus, df_tokens, df_e)
    results_raw_versions[vname] = df_raw_v
    results_cm_ibridi[vname], _ = apply_corpus_mean_norm(df_raw_v)

plot_emotion_grid(
    panels=[
        ('Raw (score originali)', df_raw),
        ('corpus_mean ItEm (score orig)',   results_cm_ibridi['Originale (α=0)']),
    ],
    total=total,
    title='Corpus_mean ItEm (Formula 3.5): confronto versioni',
).show()

### Conclusione: corpus_mean ItEm su ELIta

Corpus_mean di ItEm (Formula 3.5) **riduce aspettativa** dal **57%** (raw) al **9%**.

Dividendo per lo score raw, ogni emozione viene riportata su una scala comparabile. Le emozioni con valore medio alto nel corpus vengono penalizzate proporzionalmente alla loro diffusione.

## Diagnosi del bias - top driver di aspettativa

Per capire perché il raw mostra il 57% di aspettativa, identifichiamo le parole che contribuiscono di più: **frequenza × score ELIta = contributo totale**.
Il `corpus_mean` normalizza questi contributi dividendo per lo score raw.

In [10]:
POS_FILTER = ['ADJ', 'NOUN', 'VERB']
df_filt   = df_tokens[df_tokens['pos'].isin(POS_FILTER)].copy()
elita_idx = set(df_elita_orig.index)

matched = sorted(set(df_filt['lemma']).intersection(elita_idx))
freq    = df_filt[df_filt['lemma'].isin(matched)]['lemma'].value_counts()

freq_df = freq.reset_index()
freq_df.columns = ['lemma', 'frequenza']
er = df_elita_orig.loc[matched, BASIC_EMOTIONS].reset_index()
er.columns = ['lemma'] + BASIC_EMOTIONS
freq_df = freq_df.merge(er, on='lemma', how='left')
freq_df['word_sum']    = freq_df[BASIC_EMOTIONS].sum(axis=1)
freq_df['contrib_asp'] = freq_df['frequenza'] * freq_df['aspettativa']

print('Top 25 parole per contributo ad ASPETTATIVA (frequenza × score):')
display(freq_df.nlargest(25, 'contrib_asp')[
    ['lemma', 'frequenza', 'aspettativa', 'word_sum', 'contrib_asp']
].round(3).reset_index(drop=True))

Top 25 parole per contributo ad ASPETTATIVA (frequenza × score):


,lemma,frequenza,aspettativa,word_sum,contrib_asp
0,fare,1444,0.58,1.32,837.52
1,avere,922,0.58,2.86,534.76
2,anno,504,0.54,1.79,272.16
3,vedere,504,0.54,2.63,272.16
4,film,563,0.42,2.71,236.46
5,trovare,194,0.92,4.04,178.48
6,pensare,222,0.75,3.75,166.50
7,dire,404,0.38,1.93,153.52
8,dare,170,0.71,2.76,120.70
9,tempo,166,0.71,2.43,117.86


Le prime parole (*fare*, *avere*, *vedere*...) sono lemmi generici o legati al topic.
`fare` da sola vale **940 punti** (1621 occorrenze × 0.58), e insieme alle altre parole top contribuisce a spiegare il 57% di aspettativa nel raw.

Come detto precedentemente, una possibile causa è che durante la costruzione del lessico, **aspettativa** abbia assorbito il contributo di parole molto frequenti e generiche, creando un bias numerico che si riflette nel raw.

Il corpus_mean ridimensiona questi contributi: dividendo per score raw, anche le parole con alta frequenza e score elevato vengono pesate proporzionalmente al loro ruolo medio nel corpus.

## Metodo finale: corpus_mean di ItEm su tutte le versioni ELIta

Applichiamo la **corpus_mean normalisation** (Formula 3.5 di ItEm) a tutte e quattro le versioni.

Nessun filtraggio lessicale — il bias di aspettativa è gestito interamente dalla normalizzazione.

In [11]:
rows = []
for vname in MATRICES:
    c_raw = results_raw_versions[vname]['dominant_emotion'].value_counts()
    c_cm  = results_cm_ibridi[vname]['dominant_emotion'].value_counts()
    rows.append({
        'versione':    vname,
        'raw %':       round(c_raw.get('aspettativa', 0) / total * 100, 1),
        'corpus_mean %': round(c_cm.get('aspettativa', 0) / total * 100, 1),
    })

print('Effetto corpus_mean ItEm su aspettativa:')
display(pd.DataFrame(rows).set_index('versione'))

Effetto corpus_mean ItEm su aspettativa:


,raw %,corpus_mean %
versione,,
Originale (α=0),59.6,7.2
Ibrido (α=0.2),60.5,8.0
Ibrido (α=0.5),61.1,8.3
Ibrido (α=0.8),58.4,8.6


In [18]:
results_final = {}
match_rows = []
for vname, df_e in MATRICES.items():
    df_raw_v = detect_emotions(df_corpus, df_tokens, df_e)
    results_final[vname], _ = apply_corpus_mean_norm(df_raw_v)
    matched = (df_raw_v['n_tokens_matched'] > 0).sum()
    match_rows.append({'versione': vname, 'match': matched, 'totale': len(df_raw_v),
                       '%': round(matched / len(df_raw_v) * 100)})

display(pd.DataFrame(match_rows).set_index('versione'))

,match,totale,%
versione,,,
Originale (α=0),4026,4311,93
Ibrido (α=0.2),4026,4311,93
Ibrido (α=0.5),4026,4311,93
Ibrido (α=0.8),4026,4311,93


In tutte le versioni di alpha (0, 0.2, 0.5, 0.8), i token riconosciuti da ELIta sono gli stessi non si perde nulla in termini di copertura.

In [13]:
plot_emotion_grid(
    panels=list(results_final.items()),
    total=total,
    title='Metodo finale: corpus_mean ItEm (Formula 3.5)',
).show()

Coi vari valori di α, quasi tutte le emozioni si attestano su distribuzioni simili, con aspettativa intorno all'8-9%.

Spiccano gioia (che auenta sempre più con α crescente) e disgusto (il cui valore alto è sempre simile). La versione con α=0.5 sembra bilanciare meglio le emozioni, riducendo il gap tra aspettativa e le altre emozioni.

## Confronto Negative/Positive

Indicando come emozioni **positive** gioia, fiducia, aspettativa e sorpresa, e come **negative** tristezza, rabbia, paura e disgusto, vediamo il bilancio tra emozioni positive e negative nelle diverse versioni.

In [19]:
rows = []
for label, df_r in [('Raw', df_raw), ('corpus_mean ItEm', df_corpus_mean)]:
    c = df_r['dominant_emotion'].value_counts()
    rows.append({
        'configurazione': label,
        'positive %': round(sum(c.get(e, 0) for e in POSITIVE) / total * 100, 1),
        'negative %': round(sum(c.get(e, 0) for e in NEGATIVE) / total * 100, 1),
    })

display(pd.DataFrame(rows).set_index('configurazione'))

,positive %,negative %
configurazione,,
Raw,80.2,13.2
corpus_mean ItEm,46.6,46.8


Possiamo vedere che la versione `corpus_mean` con α=0 riduce il gap tra emozioni positive e negative, portando entrambe intorno al 40%. Nel raw, invece, le emozioni positive (soprattutto grazie ad aspettativa) dominano con oltre il 80%, mentre le negative sono sotto il 10%.

## Tabella riassuntiva

In [15]:
df_raw_orig = results_raw_versions['Originale (α=0)']
table_A = []
for e in BASIC_EMOTIONS:
    n_doc   = int((df_raw_orig[e] > 0).sum())
    doc_ids = set(df_raw_orig[df_raw_orig[e] > 0]['doc_id'])
    n_tok   = int(df_filt[
        df_filt['doc_id'].isin(doc_ids) &
        df_filt['lemma'].isin(set(df_elita_orig.index))
    ]['lemma'].count())
    table_A.append({'Emozione': e.capitalize(),
                    'N. Documenti (score>0)': n_doc,
                    'N. Token': n_tok})
print('Tabella A — N. documenti e token per emozione (metodo finale: corpus_mean):')
display(pd.DataFrame(table_A))

Tabella A — N. documenti e token per emozione (metodo finale: corpus_mean):


,Emozione,N. Documenti (score>0),N. Token
0,Gioia,3975,43333
1,Aspettativa,4013,43362
2,Rabbia,3955,43293
3,Disgusto,3904,43225
4,Tristezza,3955,43293
5,Sorpresa,4016,43369
6,Paura,3959,43309
7,Fiducia,4002,43357


In [16]:
dom = results_final['Originale (α=0)']['dominant_emotion'].value_counts()
tot = len(results_final['Originale (α=0)'])
table_B = [{'Emozione':e.capitalize(),
            'N. Documenti dom':int(dom.get(e,0)),
            '% totale':'{:.1f}%'.format(dom.get(e,0)/tot*100)}
           for e in BASIC_EMOTIONS+['neutrale']]
print('\nTabella B — Emozione dominante (corpus_mean ItEm, Formula 3.5):')
display(pd.DataFrame(table_B))


Tabella B — Emozione dominante (corpus_mean ItEm, Formula 3.5):


,Emozione,N. Documenti dom,% totale
0,Gioia,641,14.9%
1,Aspettativa,310,7.2%
2,Rabbia,340,7.9%
3,Disgusto,858,19.9%
4,Tristezza,454,10.5%
5,Sorpresa,519,12.0%
6,Paura,367,8.5%
7,Fiducia,537,12.5%
8,Neutrale,285,6.6%


## Conclusioni

- **Raw**: aspettativa domina (~60%). Il bias è numerico: aspettativa ha il raw score, molto più alto rispetto alle altre emozioni.
- **Rimozione aspettativa** (corpus_7emo): utile per l'esplorazione, ma non come metodo finale.
- **Corpus_mean ItEm** (Formula 3.5): aspettativa scende a **8%**. Divide ogni score per la media di corpus di quell'emozione — le emozioni sistematicamente alte vengono penalizzate proporzionalmente.
  - Il `corpus_mean` ridimensiona il bias numerico di aspettativa senza rimuovere alcuna parola dal lessico. Il residuo (~8%) riflette la genuina caratterizzazione semantica del dominio *notizie*.